In [ ]:
import pandas as pd
import plotly.graph_objects as go
from config import DATA
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
)

from lib.model_development import LightGbmModelBuilder

In [ ]:
dataset_path = DATA / "processed" / "basic_data_prep"
datasets = {f.stem: pd.read_csv(f) for f in sorted(dataset_path.glob("*.csv"))}
for k, v in datasets.items():
    print(f"{k}: \n{v}\n")

In [ ]:
model_params = {"verbose": 2}
results_dict = lgbm_model = LightGbmModelBuilder.from_split_data(
    datasets["X_train"],
    datasets["y_train"],
    datasets["X_test"],
    datasets["y_test"],
    model_params,
).run()

# Validate results

In [ ]:
y_score_test = results_dict["pred_test"]
y_true_test = results_dict["y_test"]
print(f"y_true_test: {y_true_test}")
print(f"y_score_test: {y_score_test}")
y_score_train = results_dict["pred_train"]
y_true_train = results_dict["y_train"]
print(f"y_true_train: {y_true_train}")
print(f"y_score_train: {y_score_train}")

In [ ]:
auc_test = roc_auc_score(y_true_test, y_score_test)
gini_test = 2 * auc_test - 1
auc_train = roc_auc_score(y_true_train, y_score_train)
gini_train = 2 * auc_train - 1

metrics = {
    "test auc": auc_test,
    "test gini": gini_test,
    "train auc": auc_train,
    "train gini": gini_train,
}

print(metrics)

In [ ]:
fpr_test, tpr_test, thr_test = roc_curve(y_true_test, y_score_test)
fpr_train, tpr_train, thr_train = roc_curve(y_true_train, y_score_train)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        hoverinfo="skip",
        line={"color": "#B0B4BA", "width": 1, "dash": "dash"},
        showlegend=False,
    )
)
fig.add_trace(
    go.Scatter(
        x=fpr_test,
        y=tpr_test,
        mode="lines",
        customdata=thr_test,
        line={"color": "#2B6CB0", "width": 2},
        showlegend=False,
        hovertemplate="FPR %{x:.3f}<br>TPR %{y:.3f}"
        "<br>threshold %{customdata:.3f}<extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=fpr_train,
        y=tpr_train,
        mode="lines",
        customdata=thr_train,
        line={"color": "#FF0000", "width": 2},
        showlegend=False,
        hovertemplate="FPR %{x:.3f}<br>TPR %{y:.3f}"
        "<br>threshold %{customdata:.3f}<extra></extra>",
    )
)
fig.update_layout(
    title=f"ROC — LightGBM: test (AUC {auc_test:.3f}, Gini {2 * auc_test - 1:.3f}),<br> train (AUC {auc_train:.3f}, Gini {2 * auc_train - 1:.3f})",
    xaxis={
        "title": "False positive rate",
        "range": [0, 1],
        "gridcolor": "#EDEFF2",
        "zeroline": False,
    },
    yaxis={
        "title": "True positive rate",
        "range": [0, 1],
        "gridcolor": "#EDEFF2",
        "zeroline": False,
        "scaleanchor": "x",
        "scaleratio": 1,
    },
    template="plotly_white",
    width=560,
    height=560,
    margin={"l": 60, "r": 30, "t": 60, "b": 60},
)
fig.show()

**Conclusion:** OK, this fixes the issues with discrimination at the top end but is significantly overfitting. Let's try to reduce the complexity of the LGBM model to see if we can improve.

In [ ]:
restricted_model_params = {
    "verbose": 2,
    "num_leaves": 20,
    "max_depth": 4,
    "min_child_samples": 100,
    "n_estimators": 60,
}
restricted_results_dict = lgbm_model = LightGbmModelBuilder.from_split_data(
    datasets["X_train"],
    datasets["y_train"],
    datasets["X_test"],
    datasets["y_test"],
    restricted_model_params,
).run()

In [ ]:
y_score_test = restricted_results_dict["pred_test"]
y_true_test = restricted_results_dict["y_test"]
print(f"y_true_test: {y_true_test}")
print(f"y_score_test: {y_score_test}")
y_score_train = restricted_results_dict["pred_train"]
y_true_train = restricted_results_dict["y_train"]
print(f"y_true_train: {y_true_train}")
print(f"y_score_train: {y_score_train}")

In [ ]:
auc_test = roc_auc_score(y_true_test, y_score_test)
gini_test = 2 * auc_test - 1
auc_train = roc_auc_score(y_true_train, y_score_train)
gini_train = 2 * auc_train - 1

metrics = {
    "test auc": auc_test,
    "test gini": gini_test,
    "train auc": auc_train,
    "train gini": gini_train,
}

print(metrics)

In [ ]:
fpr_test, tpr_test, thr_test = roc_curve(y_true_test, y_score_test)
fpr_train, tpr_train, thr_train = roc_curve(y_true_train, y_score_train)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        hoverinfo="skip",
        line={"color": "#B0B4BA", "width": 1, "dash": "dash"},
        showlegend=False,
    )
)
fig.add_trace(
    go.Scatter(
        x=fpr_test,
        y=tpr_test,
        mode="lines",
        customdata=thr_test,
        line={"color": "#2B6CB0", "width": 2},
        showlegend=False,
        hovertemplate="FPR %{x:.3f}<br>TPR %{y:.3f}"
        "<br>threshold %{customdata:.3f}<extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=fpr_train,
        y=tpr_train,
        mode="lines",
        customdata=thr_train,
        line={"color": "#FF0000", "width": 2},
        showlegend=False,
        hovertemplate="FPR %{x:.3f}<br>TPR %{y:.3f}"
        "<br>threshold %{customdata:.3f}<extra></extra>",
    )
)
fig.update_layout(
    title=f"ROC — Restricted LightGBM: test (AUC {auc_test:.3f}, Gini {2 * auc_test - 1:.3f}),<br> train (AUC {auc_train:.3f}, Gini {2 * auc_train - 1:.3f})",
    xaxis={
        "title": "False positive rate",
        "range": [0, 1],
        "gridcolor": "#EDEFF2",
        "zeroline": False,
    },
    yaxis={
        "title": "True positive rate",
        "range": [0, 1],
        "gridcolor": "#EDEFF2",
        "zeroline": False,
        "scaleanchor": "x",
        "scaleratio": 1,
    },
    template="plotly_white",
    width=560,
    height=560,
    margin={"l": 60, "r": 30, "t": 60, "b": 60},
)
fig.show()

OK, overfitting significantly reduced through restriction of model complexity without eroding the advantage compared to logistic too much. Hyperparameter tuning therefore probably sufficient to find acceptable model.